# 04 · Deployment (FastAPI + adapter từ Drive)

Giai đoạn 4: phục vụ model qua API `/extract-cccd/`, và **đo hiệu năng triển khai
của cả 3 kiến trúc**.

Base model (local) load 4-bit + LoRA adapter **đọc trực tiếp từ Google Drive**, dùng
ngrok để lộ public URL.

**Một base, hai adapter.** Mỗi model có adapter riêng cho mặt trước và mặt sau
(`{model_key}-cccd-lora-{front,back}`). Server nạp base 4-bit **một lần** rồi gắn cả
hai adapter (PEFT multi-adapter), mỗi ảnh chỉ `set_adapter()` theo mặt thẻ của nó —
rẻ hơn hẳn việc chạy 2 process cho 2 mặt. Mặt nào chưa train thì request cho mặt đó
bị từ chối kèm thông báo rõ ràng, thay vì âm thầm chạy bằng adapter của mặt kia.

**Phục vụ được cả 3 model** — đổi `MODEL_KEY` ở cell dưới (mỗi lần một model, vì
GPU chỉ đủ cho một base). `app/main.py` đọc `vlm_meta.json` cạnh adapter để xác nhận
base model khớp. Tiền xử lý ảnh và tham số sinh lấy từ `vlm_registry` — **giống hệt
lúc train và lúc đo metric**.

| Mục | Nội dung |
|---|---|
| 1–4 | Bật API cho `MODEL_KEY`, test 1 ảnh mỗi mặt |
| 5 | Nhiều ảnh / 1 request (`/extract-cccd/batch`), trộn cả 2 mặt |
| 6 | Đo như app thật: quét mức đồng thời + batch qua HTTP |
| 7 | **So sánh cả 3 model** + dò trần batch của GPU (không cần bật API) |
| 8 | Gộp thành một bảng so sánh duy nhất |

Đây là trục **tốc độ**; điểm số FA/CER/F1 vẫn lấy từ notebook 03/05
(`evaluate.py`, chạy tuần tự batch=1).

## 1. Mount Drive + trỏ BASE_MODEL (local) & CHECKPOINT_DIR vào Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

# ═══ CẤU HÌNH ════════════════════════════════════════════════════════════
MODEL_KEY = 'qwen'     # model đem PHỤC VỤ: 'qwen' | 'internvl' | 'llama_vision'
MAX_BATCH = 4          # số ảnh tối đa chạy chung 1 lần generate (mục 7 sẽ dò lại)
# ═════════════════════════════════════════════════════════════════════════
# Không còn biến SIDE: server nạp adapter của CẢ HAI mặt lên cùng một base và tự
# chọn theo mặt thẻ của từng ảnh. Mục 7 (bench local) đo cả 3 model nên không phụ
# thuộc MODEL_KEY ở đây.

PROJECT_DRIVE = Path('/content/drive/MyDrive/cccd_project')
DATA_DRIVE    = PROJECT_DRIVE / 'Data'
REPO_DRIVE    = DATA_DRIVE / 'label_CCCD'

sys.path.insert(0, str(REPO_DRIVE))
from src.models.vlm_registry import resolve, resolve_model_dir

spec           = resolve(MODEL_KEY)
MODELS_DIR     = DATA_DRIVE / 'models'
CHECKPOINT_DIR = DATA_DRIVE / 'checkpoints'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Nơi đặt snapshot base model do resolve_model_dir quyết định, KHÔNG hardcode:
# Llama-3.2-11B-Vision nặng ~21GB, không vừa Google Drive free (15GB) → tải nửa
# chừng rồi chết, để lại thư mục thiếu shard. Hàm này ưu tiên Drive (bền qua các
# lần Colab ngắt), nhưng rơi về SSD /content/models khi Drive không đủ chỗ, và
# luôn dùng lại nơi nào ĐÃ có snapshot đủ file.
MODEL_LOCAL = resolve_model_dir(spec, MODELS_DIR)
SSD_NOTE = '' if str(MODEL_LOCAL).startswith('/content/drive') else \
    '   ⚠ SSD tạm — mất khi ngắt phiên, notebook sau phải tải lại'

# app/main.py tự tìm {MODEL_KEY}-cccd-lora-{front,back} trong CHECKPOINT_DIR —
# đúng quy ước đặt tên của notebook 02. Không trỏ vào một adapter cụ thể nữa.
os.environ['MODEL_KEY']      = MODEL_KEY
os.environ['CHECKPOINT_DIR'] = str(CHECKPOINT_DIR)
os.environ['MODELS_DIR']     = str(MODELS_DIR)
os.environ['BASE_MODEL']     = str(MODEL_LOCAL)
# uvicorn được khởi động bằng subprocess.Popen ở mục 3 nên nó thừa hưởng env này.
os.environ['MAX_BATCH_SIZE'] = str(MAX_BATCH)
os.environ.pop('ADAPTER_DIR', None)   # cách cũ (1 adapter) — không dùng nữa

IMAGE_DIRS = [DATA_DRIVE / 'Front', DATA_DRIVE / 'Back']

print('MODEL       =', spec.model_id, f'[{spec.key}]')
print('BASE_MODEL  =', os.environ['BASE_MODEL'], SSD_NOTE)
print('CHECKPOINTS =', CHECKPOINT_DIR)
print('MAX_BATCH   =', MAX_BATCH)

# --- Kiểm kê adapter của CẢ 3 model, không chỉ model đang phục vụ ---
import json
from src.models.vlm_registry import REGISTRY
from src.utils.cccd_schema import CardSide

print('\nAdapter có trên Drive:')
for key in sorted(REGISTRY):
    row = []
    for side, token in (('truoc', 'front'), ('sau', 'back')):
        path = CHECKPOINT_DIR / f'{key}-cccd-lora-{token}'
        if (path / 'adapter_config.json').exists():
            meta_path = path / 'vlm_meta.json'
            meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
            tag = f"{meta.get('trainable_params_pct')}%" if meta else 'KHÔNG có vlm_meta.json'
            row.append(f'{side} ✓ ({tag})')
        else:
            row.append(f'{side} ✗')
    mark = '←  đang phục vụ' if key == MODEL_KEY else ''
    print(f'  {key:14s} {row[0]:34s} {row[1]:34s} {mark}')
print('\nMặt nào ✗ thì request cho mặt đó sẽ bị từ chối — train ở notebook 02 để bổ sung.')

## 2. Cài thư viện phục vụ (transformers bản ổn định, KHỚP notebook 02 lúc train)

In [ ]:
# Lấy source code (chứa thư mục app/ và src/) — đồng bộ đường dẫn với notebook 02/03.
!rm -rf /content/cccd && mkdir -p /content/cccd
!cp -r '/content/drive/MyDrive/cccd_project/Data/label_CCCD/.' /content/cccd/ 2>/dev/null || echo 'Đặt source vào /content/cccd'
%cd /content/cccd

# [ĐÃ SỬA] PIN cùng phiên bản transformers với lúc train (không cài bản 'git' dev)
# để tránh lệch chat-template / image-token làm adapter chạy sai.
#
# Mục 7 chạy CẢ 3 model trong một process nên cần bản đủ mới cho model khắt khe
# nhất (InternVL: 4.52.1) — dùng max của registry. Model nào cần bản mới hơn bản
# đang cài sẽ bị benchmark BỎ QUA kèm lý do, không làm chết cả run.
from packaging.version import Version
from src.models.vlm_registry import REGISTRY
TF_MIN = max((s.min_transformers for s in REGISTRY.values()), key=Version)
print('transformers >=', TF_MIN, '(max của cả 3 model; riêng', spec.key, 'chỉ cần',
      spec.min_transformers + ')')

!pip -q install 'transformers>={TF_MIN}' qwen-vl-utils accelerate peft bitsandbytes Pillow fastapi uvicorn python-multipart pyngrok requests

## 2b. Đảm bảo base model có ĐỦ trọng số ở local
Bắt buộc chạy trước khi bật API: model có thể nằm trên SSD `/content` (khi Drive không đủ chỗ) và đã mất sau khi ngắt phiên.


In [ ]:
# --- TẢI BASE MODEL VỀ LOCAL (đủ trọng số mới thôi) ---
# [ĐÃ SỬA] Trước đây guard bằng `if not (MODEL_LOCAL/'config.json').exists()`.
# snapshot_download tải file nhỏ (config, index, tokenizer) TRƯỚC, shard nặng SAU →
# lần tải bị đứt (hết chỗ trên Drive / Colab disconnect) vẫn để lại config.json,
# guard tưởng "đã có sẵn" nên bỏ qua, và lỗi chỉ nổ ra ở tận lúc nạp model:
#     FileNotFoundError: .../model-00001-of-00005.safetensors
# ensure_snapshot() đối chiếu model.safetensors.index.json nên biết thiếu shard nào
# (và bắt được cả shard ghi dở), rồi tải tiếp — snapshot_download resume được.
from src.models.vlm_registry import ensure_snapshot, snapshot_complete

ok, reason = snapshot_complete(MODEL_LOCAL)
print(f'Snapshot hiện có: {"đủ file" if ok else reason}  →  {MODEL_LOCAL}')

# Model gated (Llama-3.2-Vision) phải đăng nhập TRƯỚC khi tải. Đã có sẵn thì không cần.
if spec.gated and not ok:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(userdata.get('HF_TOKEN'))
        print('✓ Đã đăng nhập HuggingFace')
    except Exception as exc:
        raise SystemExit(
            f'{spec.model_id} là model gated nhưng chưa đăng nhập được ({exc}).\n'
            f'1) Vào https://huggingface.co/{spec.model_id} accept license\n'
            f'2) Tạo token tại https://huggingface.co/settings/tokens\n'
            f'3) Colab → 🔑 Secrets → thêm HF_TOKEN, bật "Notebook access"'
        )

ensure_snapshot(spec, MODEL_LOCAL)   # raise nếu tải xong mà vẫn thiếu file
print('✓ Base model sẵn sàng:', MODEL_LOCAL)


## 3. Khởi động FastAPI (nền) + ngrok public URL

In [ ]:
import subprocess, time, requests
from pyngrok import ngrok

# ngrok.set_auth_token('YOUR_TOKEN')  # nếu cần token riêng
proc = subprocess.Popen(['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'])

LOCAL_URL = 'http://localhost:8000'   # dùng cho benchmark: KHÔNG vòng qua ngrok

# [ĐÃ SỬA] Trước đây `time.sleep(60)` cứng: model 3B load 4-bit từ Drive có lúc lâu
# hơn thế (Drive chậm) → cell test phía dưới bắn vào server chưa sẵn sàng và fail
# khó hiểu; ngược lại load nhanh thì ngồi đợi thừa. Hỏi /health cho tới khi
# model_loaded=True (FastAPI chỉ nhận request sau khi lifespan load xong).
for attempt in range(60):
    try:
        if requests.get(f'{LOCAL_URL}/health', timeout=5).json().get('model_loaded'):
            print(f'✓ Model sẵn sàng sau ~{attempt * 5}s')
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError('Model không load xong sau 5 phút — xem log uvicorn ở ô output phía trên.')

tunnel = ngrok.connect(8000)
# [ĐÃ SỬA] `.public_url` là CHUỖI. Bản cũ ghép thẳng object tunnel vào f-string,
# ra repr dạng '<NgrokTunnel: "https://..." -> "http://localhost:8000">' nên URL
# gọi API bị hỏng.
PUBLIC_URL = tunnel.public_url

print('API public URL:', PUBLIC_URL)
print('Swagger UI    :', f'{PUBLIC_URL}/docs')
print('Cấu hình      :', requests.get(f'{LOCAL_URL}/health', timeout=5).json())

## 4. Test thử 1 ảnh mỗi mặt (kiểm tra adapter được chọn đúng)

In [ ]:
import requests, glob, json

# Test 1 ảnh MỖI MẶT: 'side=auto' suy từ tên file cccd_{front,back}_*, và server
# chọn adapter tương ứng. Mặt nào chưa train sẽ trả 'error' nói rõ.
served = requests.get(f'{LOCAL_URL}/health', timeout=10).json()['sides']
print('Server có adapter mặt:', served, '\n')

for image_dir in IMAGE_DIRS:
    files = sorted(glob.glob(str(image_dir / '*')))
    if not files:
        print(f'(bỏ qua {image_dir.name}: không có ảnh)'); continue
    sample = files[0]
    with open(sample, 'rb') as f:
        r = requests.post(f'{PUBLIC_URL}/extract-cccd/', files={'file': f},
                          params={'side': 'auto'}, timeout=600)
    print('─' * 70)
    print(Path(sample).name)
    print(json.dumps(r.json(), ensure_ascii=False, indent=2))

## 5. Test nhiều ảnh trong 1 request (`/extract-cccd/batch`)

Endpoint này nhận N ảnh, chạy chúng **chung một lần `generate`** (cắt thành chunk
`MAX_BATCH_SIZE` nếu gửi nhiều hơn) và trả kèm khối `timing`.

Ảnh khác mặt thẻ trộn chung được — mỗi ảnh vẫn dùng prompt của mặt nó. Ảnh hỏng
chỉ nhận `error` ở đúng vị trí của nó, không làm chết cả request.

In [ ]:
from pathlib import Path

# Trộn ảnh CẢ HAI MẶT trong cùng 1 request — server gom nhóm theo mặt, mỗi nhóm
# chạy bằng adapter của mặt đó. Đây là tình huống của app thật (người dùng chụp
# cả 2 mặt rồi gửi lên một lượt).
paths = []
for image_dir in IMAGE_DIRS:
    paths += sorted(glob.glob(str(image_dir / '*')))[: max(1, MAX_BATCH // 2)]
print('Gửi', len(paths), 'ảnh:', [Path(p).name for p in paths])

handles = [open(p, 'rb') for p in paths]
try:
    payload = [('files', (Path(p).name, h, 'image/jpeg')) for p, h in zip(paths, handles)]
    r = requests.post(f'{PUBLIC_URL}/extract-cccd/batch', files=payload,
                      params={'side': 'auto'}, timeout=600)
finally:
    for h in handles:
        h.close()

resp = r.json()
print('\nmodel :', resp.get('model_key'))
print('timing:', json.dumps(resp['timing'], ensure_ascii=False))
print('  (n_sides = 2 nghĩa là đã phải chạy ít nhất 2 lượt generate — 1 lượt/mặt)\n')
for item in resp['results']:
    if 'error' in item:
        print('✗', item['filename'], '—', item['error'])
    else:
        print(f"✓ [{item['side']}]", item['filename'], '|',
              json.dumps(item['data'], ensure_ascii=False)[:110])

## 6. Đo hiệu năng như app thật (`scripts/benchmark.py --mode http`)

Bắn nhiều ảnh vào API đang chạy và đo p50/p95/p99 + throughput, ở hai trục:

| Trục | Câu hỏi trả lời |
|---|---|
| `--concurrency 1,2,4` | Nhiều người dùng gọi cùng lúc thì latency đuôi tăng bao nhiêu |
| `--batch_sizes 1,2,4` | Gộp ảnh vào 1 request thì throughput lên bao nhiêu |

Hai điều **bắt buộc** để số đo không vô nghĩa:

1. **Đo qua `LOCAL_URL`, không qua ngrok.** Tunnel đi vòng qua server ngrok ở nước
   ngoài, cộng hàng trăm ms nhiễu mạng vào mỗi request — đó là số của đường truyền,
   không phải của model.
2. **Có `parse_ok` trong bảng làm chốt an toàn.** Nếu `parse_ok` tụt khi tăng batch
   thì padding đang sai (đường serve phải là `padding_side='left'`), lúc đó số tốc
   độ không dùng được.

Với 1 GPU, `ảnh/s` sẽ **chững lại** khi tăng concurrency — GPU bão hòa, thêm worker
chỉ làm dài hàng đợi. Muốn tăng throughput thì tăng batch, không tăng worker.

In [ ]:
RESULT_DRIVE = PROJECT_DRIVE / 'result'
RESULT_DRIVE.mkdir(parents=True, exist_ok=True)

# Report tự đặt tên theo model_key lấy từ /health → chạy lại cell này sau khi đổi
# MODEL_KEY sẽ ra file khác, và mục 8 gộp được cả 3.
# --limit 6: áp cho MỖI thư mục ảnh (6 trước + 6 sau) — đủ để p95 có ý nghĩa mà
# không chờ cả buổi. Tăng lên khi cần số ổn định hơn cho báo cáo.
!python scripts/benchmark.py --mode http \
    --url {LOCAL_URL} \
    --images "{IMAGE_DIRS[0]}" "{IMAGE_DIRS[1]}" --limit 6 \
    --concurrency 1,2,4 --batch_sizes 1,2,4 \
    --report_dir "{RESULT_DRIVE}"

## 7. So sánh CẢ 3 MODEL (`--mode local`) — **phải tắt API trước**

Chế độ `local` tự nạp model trong process của nó, lần lượt từng model và nhả VRAM
giữa mỗi lần. Nếu API vẫn đang chạy thì **model bị nạp 2 lần trên cùng GPU → OOM**.
Cell dưới tắt uvicorn trước rồi mới đo.

Đây là cell trả lời câu hỏi chính: *model nào nhanh hơn khi deploy*. Nó cũng cho
biết nên đặt `MAX_BATCH_SIZE` bằng bao nhiêu — chọn mức batch lớn nhất mà `peak VRAM`
còn cách trần GPU một khoảng an toàn và `ms/ảnh` vẫn còn giảm.

Model nào **bị bỏ qua** đều được nêu lý do rõ ràng, không làm chết cả run:

| Lý do | Cách xử lý |
|---|---|
| `snapshot: thiếu N/M shard` | Base chưa tải về máy này. Chạy `ensure_snapshot` (mục 2b) với `MODEL_KEY` đó trước. Llama-11B ~21GB phải nằm trên SSD `/content`, mất sau mỗi lần ngắt phiên |
| `cần transformers>=X` | InternVL cần `4.52.1`. Cài bản đủ mới cho **cả 3** rồi Runtime → Restart |
| `không có ảnh nào khớp adapter đang có` | Model đó chưa có adapter mặt nào khớp với ảnh đang đưa vào |

Đo xong, chạy lại **mục 3** để bật API trở lại (sửa `MAX_BATCH` ở mục 1 trước nếu
muốn đổi).

In [ ]:
# --- Tắt API để nhả GPU ---
try:
    ngrok.kill()
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()
    print('✓ Đã tắt uvicorn — GPU trống cho bench local')
except NameError:
    print('(API chưa từng bật trong phiên này — bỏ qua)')

# MODELS_DIR / CHECKPOINT_DIR lấy từ env đã đặt ở mục 1. Model nào chưa tải đủ
# trọng số hoặc thiếu adapter sẽ bị BỎ QUA kèm lý do, không làm chết cả run.
!python scripts/benchmark.py --mode local \
    --models qwen,internvl,llama_vision \
    --images "{IMAGE_DIRS[0]}" "{IMAGE_DIRS[1]}" --limit 4 \
    --batch_sizes 1,2,4 \
    --report_dir "{RESULT_DRIVE}"

## 8. Bảng so sánh hiệu năng triển khai giữa các model

Gộp **mọi** `benchmark_*.json` có trong `result/` — kể cả report chạy ở phiên Colab
khác. Điều này là cần thiết: 3 model hiếm khi đo được trong cùng một phiên (Llama-11B
phải tải xuống SSD `/content`, và mỗi kiến trúc có yêu cầu `transformers` riêng).

⚠ Khi ghép số từ nhiều phiên, **kiểm tra cột GPU giống nhau** trước khi kết luận —
T4 và L4 chênh nhau vài lần, so số đo giữa hai loại GPU là vô nghĩa.

Bảng này là số **tốc độ**. Bảng độ chính xác (FA/CER/F1) nằm ở notebook 05
(`result/model_comparison.md`). Báo cáo cuối nên đặt hai bảng cạnh nhau: model nhanh
nhất chưa chắc chính xác nhất, và đó chính là đánh đổi cần nêu.

In [ ]:
!python scripts/benchmark.py --mode merge --report_dir "{RESULT_DRIVE}"

# Hiển thị lại dưới dạng markdown thật trong notebook
from IPython.display import Markdown, display
display(Markdown((RESULT_DRIVE / 'benchmark_comparison.md').read_text(encoding='utf-8')))